# Péndulos independientes cargados: animación

Solo anima: lee `data_cartesianas_pc` (generado por `pendulo_cargado.f90`, que ya incluye la fuerza eléctrica) y la dibuja.

Formato del archivo (9 columnas): `t  x1  y1  x2  y2  F1x  F1y  F2x  F2y`, donde `F1` y `F2` son la fuerza eléctrica sobre cada lenteja.

**Cómo se ve la fuerza eléctrica** (datos calculados en el Fortran):

- **Flechas amarillas** sobre cada lenteja: dirección y magnitud de la fuerza eléctrica que siente.
- **Halo** alrededor de cada carga: crece y se intensifica cuando la fuerza es mayor.
- **Línea punteada** entre las cargas: su grosor y color (violeta → amarillo) indican la intensidad de la interacción.
- **Estela** de cada lenteja: sus últimas posiciones.
- **Gráfica inferior**: fuerza eléctrica F(t) con un marcador que sigue la animación.

Rojo = carga positiva, azul = carga negativa. Si se repelen las flechas apuntan hacia afuera; si se atraen, hacia adentro.

In [ ]:
# Activar matplotlib inline para Jupyter
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle
from IPython.display import HTML, display
from matplotlib import rcParams

# ==============================================================================
# 1. CONFIGURACIÓN GENERAL (¡Modifica tus variables aquí!)
# ==============================================================================
nombre_archivo = 'data_cartesianas_pc'

# PARÁMETROS DEL SISTEMA (deben coincidir con los del programa en Fortran):
x01 = 0.5    # Pivote izquierdo en (-x01, 0)
x02 = 0.5    # Pivote derecho en (x02, 0)
L1 = 1.0     # Longitud del péndulo izquierdo
L2 = 1.0     # Longitud del péndulo derecho

# SIGNO DE LAS CARGAS (solo se usa para el color: +1 rojo, -1 azul).
signo_q1 = +1
signo_q2 = +1

# CONTROL DE VELOCIDAD DE REPRODUCCIÓN:
# 1.0 = Tiempo real | 2.0 = Cámara rápida | 0.5 = Cámara lenta
factor_velocidad = 1.0

# CONTROL DE DURACIÓN DE LOS DATOS (Opcional):
recortar_tiempo_maximo = None  # Ejemplo: 3.0 para detenerse a los 3 segundos

# ESTÉTICA DE LA FUERZA ELÉCTRICA:
longitud_flecha_max = 0.6   # largo (en unidades del dibujo) de la flecha cuando la fuerza es máxima
largo_estela = 40           # cuántos cuadros de estela dejar (0 = sin estela)

guardar_video = True        # guardar también un MP4 (necesita ffmpeg)

# Ajuste de límite de memoria para animaciones en Jupyter (en MB)
rcParams['animation.embed_limit'] = 100.0

In [ ]:
# ==============================================================================
# 2. PROCESAMIENTO DE PIVOTES Y LECTURA DE DATOS
# ==============================================================================
pivote1_x, pivote1_y = -x01, 0.0
pivote2_x, pivote2_y = x02, 0.0

print("Leyendo datos del archivo...")
def leer_datos(archivo):
    datos = np.loadtxt(archivo)
    # columnas: t x1 y1 x2 y2 F1x F1y F2x F2y
    return [datos[:, j] for j in range(9)]

t, x1, y1, x2, y2, F1x, F1y, F2x, F2y = leer_datos(nombre_archivo)

# Aplicar recorte de tiempo si se especificó arriba
if recortar_tiempo_maximo is not None:
    m = t <= recortar_tiempo_maximo
    t, x1, y1, x2, y2, F1x, F1y, F2x, F2y = [v[m] for v in (t, x1, y1, x2, y2, F1x, F1y, F2x, F2y)]

# Magnitud de la fuerza (para escalar flechas, halos y línea)
Fabs = np.hypot(F1x, F1y)
Fmax = max(Fabs.max(), 1e-30)
print(f"-> |F| entre {Fabs.min():.3g} y {Fabs.max():.3g} N")

In [ ]:
# ==============================================================================
# 3. CÁLCULO DE TIEMPOS Y SINCRONIZACIÓN AUTOMÁTICA
# ==============================================================================
duracion_real = t[-1] - t[0]
total_puntos = len(t)

fps_objetivo = 50
frames_necesarios = max(1, int(duracion_real * fps_objetivo * factor_velocidad))
salto = max(1, total_puntos // frames_necesarios)

frames_animados = range(0, total_puntos, salto)
num_frames_finales = len(frames_animados)

intervalo_jupyter = max(1, int((duracion_real * 1000) / (num_frames_finales * factor_velocidad)))
fps_video_final = max(1, int((num_frames_finales / duracion_real) * factor_velocidad))

print(f"-> Duración de los datos a animar: {duracion_real:.2f} segundos")
print(f"-> Usando 1 de cada {salto} puntos (Total frames: {num_frames_finales})")
print(f"-> FPS del Video final: {fps_video_final} | Intervalo Jupyter: {intervalo_jupyter} ms")

In [ ]:
# ==============================================================================
# 4. CONFIGURACIÓN DEL GRÁFICO (LIENZO)
# ==============================================================================
FONDO = '#0b0f1a'
col1 = '#ff4d4d' if signo_q1 >= 0 else '#4d8dff'    # rojo = positiva, azul = negativa
col2 = '#ff4d4d' if signo_q2 >= 0 else '#4d8dff'
signo1 = '+' if signo_q1 >= 0 else '\u2212'
signo2 = '+' if signo_q2 >= 0 else '\u2212'
cmap = plt.cm.plasma

fig, (ax, ax2) = plt.subplots(2, 1, figsize=(8, 8),
                              gridspec_kw={'height_ratios': [3.2, 1]})
fig.patch.set_facecolor(FONDO)
for a_ in (ax, ax2):
    a_.set_facecolor(FONDO)
    a_.tick_params(colors='#9aa4b5')
    for sp in a_.spines.values():
        sp.set_color('#3a4256')

ancho_maximo = max(abs(pivote1_x), abs(pivote2_x)) + max(L1, L2) + 0.5
alto_maximo = max(L1, L2) + 1.0

ax.set_xlim(-ancho_maximo, ancho_maximo)
ax.set_ylim(-alto_maximo, 1.0)
ax.set_aspect('equal')
ax.set_title('Péndulos independientes cargados', color='white')
ax.grid(True, linestyle='--', alpha=0.15, color='white')

# Techo y pivotes
ax.plot([pivote1_x - 1.0, pivote2_x + 1.0], [0, 0], '-', lw=6, color='#8B4513')
ax.plot(pivote1_x, pivote1_y, 'o', color='#dddddd', markersize=8, zorder=5)
ax.plot(pivote2_x, pivote2_y, 'o', color='#dddddd', markersize=8, zorder=5)

# Estelas
estela1, = ax.plot([], [], '-', color=col1, lw=1.5, alpha=0.45, zorder=1)
estela2, = ax.plot([], [], '-', color=col2, lw=1.5, alpha=0.45, zorder=1)

# Cuerdas
linea_pendulo1, = ax.plot([], [], '-', color='#d0d6e0', lw=2, zorder=2)
linea_pendulo2, = ax.plot([], [], '-', color='#d0d6e0', lw=2, zorder=2)

# Enlace eléctrico entre las cargas (color y grosor según la fuerza)
enlace, = ax.plot([], [], linestyle=(0, (4, 3)), lw=1.5, color=cmap(0.5), zorder=2)

# Halos (dos anillos por carga)
halo1_ext = Circle((0, 0), 0.2, color=col1, alpha=0.10, zorder=2); ax.add_patch(halo1_ext)
halo1_int = Circle((0, 0), 0.1, color=col1, alpha=0.25, zorder=2); ax.add_patch(halo1_int)
halo2_ext = Circle((0, 0), 0.2, color=col2, alpha=0.10, zorder=2); ax.add_patch(halo2_ext)
halo2_int = Circle((0, 0), 0.1, color=col2, alpha=0.25, zorder=2); ax.add_patch(halo2_int)

# Lentejas y signo de la carga
lenteja1, = ax.plot([], [], 'o', markersize=18, color=col1, markeredgecolor='white', zorder=3)
lenteja2, = ax.plot([], [], 'o', markersize=18, color=col2, markeredgecolor='white', zorder=3)
txt_q1 = ax.text(0, 0, signo1, color='white', ha='center', va='center',
                 fontsize=11, fontweight='bold', zorder=4)
txt_q2 = ax.text(0, 0, signo2, color='white', ha='center', va='center',
                 fontsize=11, fontweight='bold', zorder=4)

# Flechas de la fuerza eléctrica sobre cada lenteja
flechas = ax.quiver([x1[0], x2[0]], [y1[0], y2[0]], [0, 0], [0, 0],
                    angles='xy', scale_units='xy', scale=1,
                    color='#ffd166', width=0.008, headwidth=4, headlength=4.5, zorder=6)

# Textos en pantalla
tiempo_texto = ax.text(0.03, 0.94, '', transform=ax.transAxes, fontsize=12, color='white')
fuerza_texto = ax.text(0.03, 0.88, '', transform=ax.transAxes, fontsize=11, color='#ffd166')
ax.text(0.97, 0.94, 'flecha = fuerza eléctrica\nhalo/línea = intensidad',
        transform=ax.transAxes, fontsize=8, color='#9aa4b5', ha='right', va='top')

# Gráfica inferior: F(t) con marcador móvil
ax2.plot(t, Fabs, color='#ffd166', lw=1.3)
ax2.axhline(0, color='#3a4256', lw=0.8)
ax2.set_xlim(t[0], t[-1])
ax2.set_xlabel('Tiempo (s)', color='#9aa4b5')
ax2.set_ylabel('Fuerza eléctrica |F| (N)', color='#9aa4b5')
ax2.grid(True, linestyle='--', alpha=0.15, color='white')
marcador_t = ax2.axvline(t[0], color='white', lw=1, alpha=0.7)
marcador_p, = ax2.plot([t[0]], [Fabs[0]], 'o', color='white', markersize=6)

fig.tight_layout()

In [ ]:
# ==============================================================================
# 5. FUNCIONES DE ANIMACIÓN Y RENDERIZADO
# ==============================================================================
def animar(i):
    # Cuerdas y lentejas
    linea_pendulo1.set_data([pivote1_x, x1[i]], [pivote1_y, y1[i]])
    linea_pendulo2.set_data([pivote2_x, x2[i]], [pivote2_y, y2[i]])
    lenteja1.set_data([x1[i]], [y1[i]])
    lenteja2.set_data([x2[i]], [y2[i]])
    txt_q1.set_position((x1[i], y1[i]))
    txt_q2.set_position((x2[i], y2[i]))

    # Intensidad relativa de la fuerza (0 a 1)
    r = Fabs[i] / Fmax

    # Halos: crecen con la fuerza
    for h_int, h_ext, xc, yc in ((halo1_int, halo1_ext, x1[i], y1[i]),
                                 (halo2_int, halo2_ext, x2[i], y2[i])):
        h_int.set_center((xc, yc)); h_int.set_radius(0.08 + 0.12 * r)
        h_ext.set_center((xc, yc)); h_ext.set_radius(0.14 + 0.24 * r)
        h_int.set_alpha(0.15 + 0.35 * r)
        h_ext.set_alpha(0.05 + 0.18 * r)

    # Enlace eléctrico: color y grosor según la fuerza
    enlace.set_data([x1[i], x2[i]], [y1[i], y2[i]])
    enlace.set_color(cmap(0.15 + 0.85 * r))
    enlace.set_linewidth(0.8 + 4.5 * r)

    # Flechas de fuerza (longitud proporcional a la fuerza)
    flechas.set_offsets(np.c_[[x1[i], x2[i]], [y1[i], y2[i]]])
    flechas.set_UVC([longitud_flecha_max * F1x[i] / Fmax, longitud_flecha_max * F2x[i] / Fmax],
                    [longitud_flecha_max * F1y[i] / Fmax, longitud_flecha_max * F2y[i] / Fmax])

    # Estelas
    if largo_estela > 0:
        sl = slice(max(0, i - largo_estela * salto), i + 1, salto)
        estela1.set_data(x1[sl], y1[sl])
        estela2.set_data(x2[sl], y2[sl])

    # Textos y gráfica inferior
    tiempo_texto.set_text(f"t = {t[i]:.2f} s")
    fuerza_texto.set_text(f"|F| = {Fabs[i]:.3g} N")
    marcador_t.set_xdata([t[i], t[i]])
    marcador_p.set_data([t[i]], [Fabs[i]])

    return (linea_pendulo1, linea_pendulo2, lenteja1, lenteja2, txt_q1, txt_q2,
            halo1_int, halo1_ext, halo2_int, halo2_ext, enlace, flechas,
            estela1, estela2, tiempo_texto, fuerza_texto, marcador_t, marcador_p)

print("Preparando animación...")
ani = animation.FuncAnimation(
    fig, animar, frames=frames_animados, interval=intervalo_jupyter, blit=True
)
plt.close(fig)

In [ ]:
# ==============================================================================
# 6. MOSTRAR Y GUARDAR
# Intenta generar un MP4 (ligero) y lo incrusta en el notebook. Si no hay ffmpeg,
# usa el reproductor JavaScript (más pesado) y trata de guardar un GIF.
# ==============================================================================
import base64

mostrado = False
if guardar_video:
    print("Guardando archivo de video MP4 (esto puede tardar un poco)...")
    try:
        ani.save('pendulos_cargados.mp4', writer='ffmpeg', fps=fps_video_final, dpi=150,
                 savefig_kwargs={'facecolor': FONDO})
        print("¡Listo! Video guardado como 'pendulos_cargados.mp4'")
        with open('pendulos_cargados.mp4', 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        display(HTML(f'<video controls loop autoplay width="700" '
                     f'src="data:video/mp4;base64,{b64}"></video>'))
        mostrado = True
    except Exception as e:
        print("No se pudo generar el MP4 (¿está instalado ffmpeg?):", e)

if not mostrado:
    print("Mostrando con el reproductor JavaScript...")
    display(HTML(ani.to_jshtml()))
    try:
        ani.save('pendulos_cargados.gif', writer='pillow', fps=min(fps_video_final, 30),
                 savefig_kwargs={'facecolor': FONDO})
        print("GIF guardado como 'pendulos_cargados.gif'")
    except Exception as e2:
        print("No se pudo guardar el GIF:", e2)